In [1]:
import pandas as pd
import numpy as np
import os, warnings, time
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               AdaBoostClassifier, GradientBoostingClassifier,
                               BaggingClassifier, VotingClassifier)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for tmux/server use
import matplotlib.pyplot as plt
import seaborn as sns

In [43]:
#BASE = os.path.expanduser('~/hdp_project')
#os.makedirs(f'{BASE}/results', exist_ok=True)
"""
DATASETS = {
    'Dataset1': (f'{BASE}/data/heart.csv',                                     'target'),
    'Dataset2': (f'{BASE}/data/heart_2020_cleaned.csv',                        'HadHeartAttack'),
    'Dataset3': (f'{BASE}/data/heart_cleveland_upload.csv',                    'condition'),
    'Dataset4': (f'{BASE}/data/heart_disease_health_indicators_BRFSS2015.csv', 'HeartDiseaseorAttack'),
    'Dataset5': (f'{BASE}/data/heart_disease_uci.csv',                         'num'),
}
"""
DATASETS = {
    'Dataset1': ('heart.csv',                                     'target'),
    'Dataset2': ('heart_2020_cleaned.csv',                        'HadHeartAttack'),
    'Dataset3': ('heart_cleveland_upload.csv',                    'condition'),
    'Dataset4': ('heart_disease_health_indicators_BRFSS2015.csv', 'HeartDiseaseorAttack'),
    'Dataset5': ('heart_disease_uci.csv',                         'num')
}


In [44]:
# 1. PREPROCESSING
# ═══════════════════════════════════════════════════════════
def preprocess(df, target_col, encoding='ordinal', imputation='knn'):
    df = df.drop(columns=[c for c in ['id', 'dataset'] if c in df.columns])
    df = df.dropna(subset=[target_col])
    X  = df.drop(columns=[target_col])
    y  = df[target_col].copy()

    if y.nunique() > 2:
        y = (y > 0).astype(int)
    if y.dtype == object:
        y = (y == y.unique()[0]).astype(int)

    bool_cols = X.select_dtypes(include='bool').columns.tolist()
    if bool_cols:
        X[bool_cols] = X[bool_cols].astype(int)

    cat_cols = X.select_dtypes(include='object').columns.tolist()

    # NOVELTY: One-Hot vs Ordinal Encoding (Paper Section 3.2)
    if encoding == 'onehot' and cat_cols:
        X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    elif cat_cols:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[cat_cols] = enc.fit_transform(X[cat_cols])

    # NOVELTY: Mean / Median / KNN Imputation (Paper Section 3.2)
    if imputation == 'mean':
        imputer = SimpleImputer(strategy='mean')
    elif imputation == 'median':
        imputer = SimpleImputer(strategy='median')
    else:
        imputer = KNNImputer(n_neighbors=5)

    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    X = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)
    return X, y.reset_index(drop=True)


In [45]:
# 2. FEATURE SELECTION (Paper Section 3.3)
# ═══════════════════════════════════════════════════════════
def get_feature_sets(X, y):
    k = min(10, X.shape[1])
    return {
        'No_FS': X.values,
        'RFE':   RFE(RandomForestClassifier(n_estimators=50, random_state=42),
                     n_features_to_select=k).fit_transform(X, y),
        'PCA':   PCA(n_components=0.95, random_state=42).fit_transform(X),
        'UVFS':  SelectKBest(f_classif, k=k).fit_transform(X, y),
    }


In [46]:
# 3. MODELS (Paper Section 3.4)
# ═══════════════════════════════════════════════════════════
def get_models():
    return {
        'LR':   LogisticRegression(max_iter=1000, random_state=42),
        'LDA':  LinearDiscriminantAnalysis(),
        'KNN':  KNeighborsClassifier(),
        'GNB':  GaussianNB(),
        'CART': DecisionTreeClassifier(random_state=42),
        'SVM':  SVC(probability=True, random_state=42),
        'BDT':  BaggingClassifier(random_state=42),
        'RF':   RandomForestClassifier(random_state=42),
        'ET':   ExtraTreesClassifier(random_state=42),
        'AB':   AdaBoostClassifier(random_state=42, algorithm='SAMME'),
        'SGB':  GradientBoostingClassifier(random_state=42),
    }



In [47]:
# 4. NOVELTY: VOTING ENSEMBLE CLASSIFIER (Paper Section 3.4.12)
# ═══════════════════════════════════════════════════════════
def get_voting_classifiers():
    estimators = [
        ('ET',  ExtraTreesClassifier(n_estimators=100, random_state=42)),
        ('RF',  RandomForestClassifier(n_estimators=100, random_state=42)),
        ('SGB', GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('SVM', SVC(probability=True, random_state=42)),
        ('LR',  LogisticRegression(max_iter=1000, random_state=42)),
    ]
    return {
        'VEC_Hard': VotingClassifier(estimators=estimators, voting='hard'),
        'VEC_Soft': VotingClassifier(estimators=estimators, voting='soft'),
    }


In [48]:
# 5. EVALUATION
# ═══════════════════════════════════════════════════════════
def evaluate(models, X, y, cv=5):
    skf  = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    rows = []
    for name, model in models.items():
        try:
            sc = cross_validate(model, X, y, cv=skf,
                                scoring=['accuracy', 'precision', 'recall', 'f1'],
                                n_jobs=-1)
            rows.append({
                'Model':     name,
                'Accuracy':  round(sc['test_accuracy'].mean(),  4),
                'Precision': round(sc['test_precision'].mean(), 4),
                'Recall':    round(sc['test_recall'].mean(),    4),
                'F1':        round(sc['test_f1'].mean(),        4),
            })
        except Exception as e:
            print(f"    ⚠ {name} skipped: {e}")
    return pd.DataFrame(rows).sort_values('Accuracy', ascending=False)



In [49]:
# 6. HYPERPARAMETER TUNING (Paper Section 3.6)
# ═══════════════════════════════════════════════════════════
PARAM_GRIDS = {
    'RF':   {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]},
    'ET':   {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]},
    'SVM':  {'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear']},
    'KNN':  {'n_neighbors': [3, 5, 7, 11], 'weights': ['uniform', 'distance']},
    'CART': {'max_depth': [None, 5, 10, 20], 'min_samples_split': [2, 5]},
    'LR':   {'C': [0.01, 0.1, 1, 10]},
    'SGB':  {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]},
    'AB':   {'n_estimators': [50, 100, 200]},
    'BDT':  {'n_estimators': [10, 50, 100]},
    'GNB':  {}, 'LDA': {},
}

def tune_model(model_name, model, X, y, cv=5):
    grid = PARAM_GRIDS.get(model_name, {})
    if not grid:
        return model, {}
    gs = GridSearchCV(model, grid, cv=cv, scoring='accuracy', n_jobs=-1)
    gs.fit(X, y)
    return gs.best_estimator_, gs.best_params_



In [50]:
# EXPERIMENT 1 — Individual Models × Feature Selection
# ═══════════════════════════════════════════════════════════
def run_experiment1():
    print("\n" + "="*60)
    print("EXPERIMENT 1: Individual Models × Feature Selection Methods")
    print("="*60)

    all_results    = []
    tuning_results = []

    for ds_name, (path, target) in DATASETS.items():
        print(f"\n📂 {ds_name}")
        df = pd.read_csv(path)
        if len(df) > 50000:
            df = df.sample(50000, random_state=42)
            print(f"  ⚠ Subsampled to 50,000 rows")
        X, y = preprocess(df, target)
        print(f"  Shape: {X.shape} | Balance: {y.value_counts().to_dict()}")
        feature_sets = get_feature_sets(X, y)

        for fs_name, X_fs in feature_sets.items():
            print(f"  → {fs_name} ...", end=' ', flush=True)
            res = evaluate(get_models(), X_fs, y)
            res['Dataset']           = ds_name
            res['Feature_Selection'] = fs_name
            all_results.append(res)
            print(f"Best: {res.iloc[0]['Model']} @ {res.iloc[0]['Accuracy']:.4f}")

        # Tune best model
        print(f"  🔧 Tuning ...", end=' ', flush=True)
        best_name  = all_results[-len(feature_sets)]['Model'].iloc[0]
        best_model = get_models()[best_name]
        X_nofs     = feature_sets['No_FS']
        tuned_model, best_params = tune_model(best_name, best_model, X_nofs, y)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        sc  = cross_validate(tuned_model, X_nofs, y, cv=skf,
                             scoring=['accuracy', 'precision', 'recall', 'f1'],
                             n_jobs=-1)
        tuning_results.append({
            'Dataset':   ds_name,
            'Model':     best_name,
            'Params':    str(best_params),
            'Accuracy':  round(sc['test_accuracy'].mean(),  4),
            'Precision': round(sc['test_precision'].mean(), 4),
            'Recall':    round(sc['test_recall'].mean(),    4),
            'F1':        round(sc['test_f1'].mean(),        4),
        })
        print(f"Tuned {best_name} @ {sc['test_accuracy'].mean():.4f}")

    final_df  = pd.concat(all_results, ignore_index=True)
    tuning_df = pd.DataFrame(tuning_results)
    final_df.to_csv(f'{BASE}/results/all_results.csv', index=False)
    tuning_df.to_csv(f'{BASE}/results/tuning_results.csv', index=False)
    print("\n✅ Experiment 1 complete! Saved: all_results.csv, tuning_results.csv")
    return final_df, tuning_df


In [51]:
# NOVELTY A — Voting Ensemble Classifier (VEC)
# ═══════════════════════════════════════════════════════════
def run_novelty_vec(final_df):
    print("\n" + "="*60)
    print("NOVELTY A: Voting Ensemble Classifier — Paper Sec 3.4.12")
    print("="*60)

    vec_results = []
    for ds_name, (path, target) in DATASETS.items():
        print(f"\n📂 {ds_name}")
        df = pd.read_csv(path)
        if len(df) > 50000:
            df = df.sample(50000, random_state=42)
        X, y = preprocess(df, target)
        feature_sets = get_feature_sets(X, y)

        for fs_name, X_fs in feature_sets.items():
            print(f"  → {fs_name} ...", end=' ', flush=True)
            res = evaluate(get_voting_classifiers(), X_fs, y)
            res['Dataset']           = ds_name
            res['Feature_Selection'] = fs_name
            vec_results.append(res)
            print(f"Best VEC: {res.iloc[0]['Model']} @ {res.iloc[0]['Accuracy']:.4f}")

    vec_df = pd.concat(vec_results, ignore_index=True)
    vec_df.to_csv(f'{BASE}/results/VEC_results.csv', index=False)

    print("\n--- VEC vs Best Individual Model ---")
    for ds_name in DATASETS:
        ind_best = final_df[final_df['Dataset'] == ds_name]['Accuracy'].max()
        vec_best = vec_df[vec_df['Dataset'] == ds_name]['Accuracy'].max()
        winner   = "VEC ✅" if vec_best >= ind_best else "Individual ✅"
        print(f"  {ds_name}: Individual={ind_best:.4f} | VEC={vec_best:.4f} | Winner: {winner}")

    print("\n✅ Novelty A complete! Saved: VEC_results.csv")
    return vec_df



In [52]:
# NOVELTY B — Imputation Strategy Comparison  ← BUG FIXED HERE
# ═══════════════════════════════════════════════════════════
def run_novelty_imputation():
    print("\n" + "="*60)
    print("NOVELTY B: Imputation Strategy Comparison — Paper Sec 3.2")
    print("="*60)

    imp_rows = []
    for ds_name, (path, target) in DATASETS.items():
        df = pd.read_csv(path)
        if len(df) > 50000:
            df = df.sample(50000, random_state=42)
        row = {'Dataset': ds_name}

        for strategy in ['mean', 'median', 'knn']:
            X_imp, y_imp = preprocess(df, target, imputation=strategy)
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

            # FIX: pass scoring as LIST so key becomes 'test_accuracy' not 'test_score'
            sc = cross_validate(
                RandomForestClassifier(random_state=42),
                X_imp, y_imp,
                cv=skf,
                scoring=['accuracy'],   # <-- LIST not string
                n_jobs=-1
            )
            row[f'Imputation_{strategy}'] = round(sc['test_accuracy'].mean(), 4)

        imp_rows.append(row)
        print(f"  {ds_name}: "
              f"mean={row['Imputation_mean']} | "
              f"median={row['Imputation_median']} | "
              f"knn={row['Imputation_knn']}")

    imp_df = pd.DataFrame(imp_rows).set_index('Dataset')
    imp_df.to_csv(f'{BASE}/results/Imputation_Comparison.csv')
    print("\nImputation Comparison Table:")
    print(imp_df)
    print("\n✅ Novelty B complete! Saved: Imputation_Comparison.csv")
    return imp_df



In [53]:
# NOVELTY C — Encoding Strategy Comparison
# ═══════════════════════════════════════════════════════════
def run_novelty_encoding():
    print("\n" + "="*60)
    print("NOVELTY C: Encoding Strategy Comparison — Paper Sec 3.2")
    print("="*60)

    enc_rows = []
    for ds_name, (path, target) in DATASETS.items():
        df = pd.read_csv(path)
        if len(df) > 50000:
            df = df.sample(50000, random_state=42)
        row = {'Dataset': ds_name}

        for enc_type in ['ordinal', 'onehot']:
            X_enc, y_enc = preprocess(df, target, encoding=enc_type)
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

            # FIX: pass scoring as LIST
            sc = cross_validate(
                RandomForestClassifier(random_state=42),
                X_enc, y_enc,
                cv=skf,
                scoring=['accuracy'],   # <-- LIST not string
                n_jobs=-1
            )
            row[f'Encoding_{enc_type}'] = round(sc['test_accuracy'].mean(), 4)

        enc_rows.append(row)
        print(f"  {ds_name}: "
              f"ordinal={row['Encoding_ordinal']} | "
              f"onehot={row['Encoding_onehot']}")

    enc_df = pd.DataFrame(enc_rows).set_index('Dataset')
    enc_df.to_csv(f'{BASE}/results/Encoding_Comparison.csv')
    print("\nEncoding Comparison Table:")
    print(enc_df)
    print("\n✅ Novelty C complete! Saved: Encoding_Comparison.csv")
    return enc_df



In [54]:
# PLOTS
# ═══════════════════════════════════════════════════════════
def generate_plots(final_df, vec_df, imp_df, enc_df):
    print("\n" + "="*60)
    print("GENERATING ALL PLOTS")
    print("="*60)

    # Fig 1: Accuracy Bar Charts (paper Fig 3-6)
    fig, axes = plt.subplots(2, 2, figsize=(22, 12))
    for ax, fs in zip(axes.flatten(), ['No_FS', 'RFE', 'PCA', 'UVFS']):
        subset = final_df[final_df['Feature_Selection'] == fs]
        pivot  = subset.pivot_table(index='Model', columns='Dataset', values='Accuracy')
        pivot.plot(kind='bar', ax=ax, title=f'Accuracy — {fs}', ylim=(0, 1))
        ax.set_ylabel('Accuracy')
        ax.tick_params(axis='x', rotation=45)
        ax.legend(fontsize=7, loc='lower right')
        ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.suptitle('ML Model Accuracy Across Datasets and Feature Selection Methods',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{BASE}/results/Fig1_Accuracy_Comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✅ Fig1_Accuracy_Comparison.png")

    # Fig 2: Heatmap
    pivot_heat = final_df.pivot_table(index='Model',
                                       columns=['Dataset', 'Feature_Selection'],
                                       values='Accuracy', aggfunc='max')
    plt.figure(figsize=(24, 6))
    sns.heatmap(pivot_heat, annot=True, fmt='.3f', cmap='YlGn',
                linewidths=0.5, cbar_kws={'label': 'Accuracy'})
    plt.title('Accuracy Heatmap — All Models × Datasets × Feature Methods')
    plt.tight_layout()
    plt.savefig(f'{BASE}/results/Fig2_Heatmap.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✅ Fig2_Heatmap.png")

    # Fig 3: VEC vs Individual
    ds_names  = list(DATASETS.keys())
    ind_bests = [final_df[final_df['Dataset'] == ds]['Accuracy'].max() for ds in ds_names]
    vec_bests = [vec_df[vec_df['Dataset'] == ds]['Accuracy'].max() for ds in ds_names]
    x, width  = np.arange(len(ds_names)), 0.35
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - width/2, ind_bests, width, label='Best Individual', color='#4472C4')
    ax.bar(x + width/2, vec_bests, width, label='VEC',             color='#ED7D31')
    ax.set_xticks(x); ax.set_xticklabels(ds_names, rotation=15)
    ax.set_ylim(0.7, 1.05); ax.set_ylabel('Accuracy')
    ax.set_title('Individual Models vs Voting Ensemble Classifier (VEC)')
    ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.5)
    for i, (iv, vv) in enumerate(zip(ind_bests, vec_bests)):
        ax.text(i - width/2, iv + 0.005, f'{iv:.3f}', ha='center', fontsize=8)
        ax.text(i + width/2, vv + 0.005, f'{vv:.3f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{BASE}/results/Fig3_VEC_vs_Individual.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✅ Fig3_VEC_vs_Individual.png")

    # Fig 4: Imputation Comparison
    imp_df.plot(kind='bar', figsize=(12, 5),
                title='Imputation Strategy Comparison (RF Accuracy)',
                color=['#4472C4', '#ED7D31', '#A9D18E'], ylim=(0.5, 1.05))
    plt.ylabel('Accuracy'); plt.xticks(rotation=15)
    plt.legend(['Mean', 'Median', 'KNN'])
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'{BASE}/results/Fig4_Imputation_Comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✅ Fig4_Imputation_Comparison.png")

    # Fig 5: Encoding Comparison
    enc_df.plot(kind='bar', figsize=(10, 5),
                title='Encoding Strategy Comparison (RF Accuracy)',
                color=['#4472C4', '#ED7D31'], ylim=(0.5, 1.05))
    plt.ylabel('Accuracy'); plt.xticks(rotation=15)
    plt.legend(['Ordinal', 'One-Hot'])
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'{BASE}/results/Fig5_Encoding_Comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✅ Fig5_Encoding_Comparison.png")

    # Fig 6: Confusion Matrices
    best_model_map = {
        'Dataset1': ('heart.csv',                                     'target',               DecisionTreeClassifier(random_state=42)),
        'Dataset2': ('heart_2020_cleaned.csv',                        'HadHeartAttack',       GradientBoostingClassifier(random_state=42)),
        'Dataset3': ('heart_cleveland_upload.csv',                    'condition',            LinearDiscriminantAnalysis()),
        'Dataset4': ('heart_disease_health_indicators_BRFSS2015.csv', 'HeartDiseaseorAttack', LogisticRegression(max_iter=1000)),
        'Dataset5': ('heart_disease_uci.csv',                         'num',                  SVC(random_state=42)),
    }
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    for ax, (ds_name, (fname, target, model)) in zip(axes, best_model_map.items()):
        df = pd.read_csv(f'{BASE}/data/{fname}')
        if len(df) > 50000:
            df = df.sample(50000, random_state=42)
        X, y = preprocess(df, target)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                               random_state=42, stratify=y)
        model.fit(Xtr, ytr)
        cm = confusion_matrix(yte, model.predict(Xte))
        ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease']).plot(
            ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f'{ds_name}\n({type(model).__name__})', fontsize=8)
    plt.suptitle('Confusion Matrices — Best Model Per Dataset', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{BASE}/results/Fig6_ConfusionMatrices.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✅ Fig6_ConfusionMatrices.png")


In [ ]:
# MAIN
# ═══════════════════════════════════════════════════════════
if __name__ == '__main__':
    start = time.time()

    print("\n" + "█"*60)
    print("  HEART DISEASE PREDICTION — FULL EXPERIMENT")
    print("█"*60)

    final_df,  tuning_df = run_experiment1()
    vec_df               = run_novelty_vec(final_df)
    imp_df               = run_novelty_imputation()
    enc_df               = run_novelty_encoding()
    generate_plots(final_df, vec_df, imp_df, enc_df)

    elapsed = time.time() - start
    print("\n" + "█"*60)
    print(f"  ALL DONE in {elapsed/60:.1f} minutes")
    print("█"*60)
    print(f"\nResults saved to: {BASE}/results/")
    print("  all_results.csv")
    print("  tuning_results.csv")
    print("  VEC_results.csv")
    print("  Imputation_Comparison.csv")
    print("  Encoding_Comparison.csv")
    print("  Fig1_Accuracy_Comparison.png")
    print("  Fig2_Heatmap.png")
    print("  Fig3_VEC_vs_Individual.png")
    print("  Fig4_Imputation_Comparison.png")
    print("  Fig5_Encoding_Comparison.png")
    print("  Fig6_ConfusionMatrices.png")


████████████████████████████████████████████████████████████
  HEART DISEASE PREDICTION — FULL EXPERIMENT
████████████████████████████████████████████████████████████

EXPERIMENT 1: Individual Models × Feature Selection Methods

📂 Dataset1
  Shape: (1025, 13) | Balance: {1: 526, 0: 499}
  → No_FS ... Best: CART @ 1.0000
  → RFE ... 

/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorith

Best: CART @ 0.9961
  → PCA ... 

/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorith

Best: CART @ 0.9961
Best: CART @ 1.0000
  🔧 Tuning ... 

/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorith

Tuned CART @ 1.0000

📂 Dataset2
  ⚠ Subsampled to 50,000 rows
  Shape: (50000, 39) | Balance: {1: 47332, 0: 2668}
  → No_FS ... 